# Character-Level Name Generation using RNN Variants
This notebook implements and compares three Recurrent Neural Network architectures for character-level name generation.
## Setup and Data Processing

In [37]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import random
import os

# Load Dataset
DATASET_PATH = 'TrainingNames.txt'

with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    training_names = [line.strip() for line in f.readlines() if line.strip()]
    
print(f"Loaded {len(training_names)} names from {DATASET_PATH}")

# Vocabulary Class
class CharVocabulary:
    def __init__(self, names):
        self.chars = sorted(list(set(''.join(names))))
        self.chars.insert(0, '<EOS>') # End of Sequence
        self.chars.insert(1, '<SOS>') # Start of Sequence
        self.char2idx = {c: i for i, c in enumerate(self.chars)}
        self.idx2char = {i: c for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars)

    def encode(self, name):
        return [self.char2idx['<SOS>']] + [self.char2idx[c] for c in name] + [self.char2idx['<EOS>']]

    def decode(self, indices):
        return ''.join([self.idx2char[i] for i in indices if i not in (0, 1)])

vocab = CharVocabulary(training_names)
print(f"Vocabulary Size: {vocab.vocab_size} characters")

# Utility Functions
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def evaluate_metrics(generated_names, training_names):
    gen_set = set(generated_names)
    train_set = set(training_names)
    novelty = (len(gen_set - train_set) / len(generated_names)) * 100 if generated_names else 0
    diversity = (len(gen_set) / len(generated_names)) * 100 if generated_names else 0
    return novelty, diversity

Loaded 1000 names from TrainingNames.txt
Vocabulary Size: 47 characters


## 1. Vanilla Recurrent Neural Network (RNN)

### (iii) Hyperparameters
* **Hidden Size**: 64
* **Number of Layers**: 1
* **Learning Rate**: 0.005

In [38]:
class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Parameter(torch.randn(vocab_size, hidden_size) * 0.1)
        self.W_ih = nn.Parameter(torch.randn(hidden_size, hidden_size) / np.sqrt(hidden_size))
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) / np.sqrt(hidden_size))
        self.b_h = nn.Parameter(torch.zeros(hidden_size))
        self.W_ho = nn.Parameter(torch.randn(hidden_size, vocab_size) / np.sqrt(hidden_size))
        self.b_o = nn.Parameter(torch.zeros(vocab_size))

    def init_hidden(self, batch_size, device):
        return torch.zeros(batch_size, self.hidden_size, device=device)

    def recurrent_step(self, x_t, h_prev):
        return torch.tanh(x_t @ self.W_ih + h_prev @ self.W_hh + self.b_h)

    def forward(self, x):
        batch_size, seq_len = x.shape
        h = self.init_hidden(batch_size, x.device)
        logits = []

        for t in range(seq_len):
            x_t = F.embedding(x[:, t], self.embedding)
            h = self.recurrent_step(x_t, h)
            logits.append((h @ self.W_ho + self.b_o).unsqueeze(1))

        return torch.cat(logits, dim=1)

    def init_generation_state(self, batch_size, device):
        return self.init_hidden(batch_size, device)

    def sample_step(self, x, state):
        x_t = F.embedding(x.squeeze(1), self.embedding)
        h = self.recurrent_step(x_t, state)
        logits = (h @ self.W_ho + self.b_o).unsqueeze(1)
        return logits, h

HIDDEN_SIZE = 64
vanilla_model = VanillaRNN(vocab.vocab_size, HIDDEN_SIZE)
print(f"(ii) Vanilla RNN Trainable Parameters: {count_parameters(vanilla_model)}")

(ii) Vanilla RNN Trainable Parameters: 14319


### Vanilla RNN: Qualitative Analysis
* **Realism**: Tends to learn basic character transitions (e.g., consonant followed by a vowel) but often loses track of the overall name structure, especially for longer names.
* **Failure Modes**: Prone to repetitive loops (e.g., "Anananan") or abrupt, nonsensical endings because it suffers from the vanishing gradient problem, forgetting early context.

## 2. Bidirectional LSTM (BLSTM)

### (iii) Hyperparameters
* **Hidden Size**: 64
* **Number of Layers**: 1
* **Learning Rate**: 0.005
* **Bidirectional**: True

In [39]:
class ManualLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.W = nn.Parameter(torch.randn(input_size, hidden_size * 4) / np.sqrt(input_size))
        self.U = nn.Parameter(torch.randn(hidden_size, hidden_size * 4) / np.sqrt(hidden_size))
        self.b = nn.Parameter(torch.zeros(hidden_size * 4))

    def forward(self, x_t, state):
        h_prev, c_prev = state
        gates = x_t @ self.W + h_prev @ self.U + self.b
        i, f, g, o = gates.chunk(4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        g = torch.tanh(g)
        o = torch.sigmoid(o)
        c = f * c_prev + i * g
        h = o * torch.tanh(c)
        return h, c

class BLSTM(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Parameter(torch.randn(vocab_size, hidden_size) * 0.1)
        self.forward_cell = ManualLSTMCell(hidden_size, hidden_size)
        self.backward_cell = ManualLSTMCell(hidden_size, hidden_size)
        self.W_out = nn.Parameter(torch.randn(hidden_size * 2, vocab_size) / np.sqrt(hidden_size * 2))
        self.b_out = nn.Parameter(torch.zeros(vocab_size))

    def init_state(self, batch_size, device):
        zeros = torch.zeros(batch_size, self.hidden_size, device=device)
        return zeros.clone(), zeros.clone()

    def forward(self, x):
        batch_size, seq_len = x.shape
        embeddings = F.embedding(x, self.embedding)

        forward_states = []
        h_f, c_f = self.init_state(batch_size, x.device)
        for t in range(seq_len):
            h_f, c_f = self.forward_cell(embeddings[:, t, :], (h_f, c_f))
            forward_states.append(h_f)

        backward_states = [None] * seq_len
        h_b, c_b = self.init_state(batch_size, x.device)
        for t in range(seq_len - 1, -1, -1):
            h_b, c_b = self.backward_cell(embeddings[:, t, :], (h_b, c_b))
            backward_states[t] = h_b

        logits = []
        for t in range(seq_len):
            combined = torch.cat([forward_states[t], backward_states[t]], dim=1)
            logits.append((combined @ self.W_out + self.b_out).unsqueeze(1))

        return torch.cat(logits, dim=1)

    def init_generation_state(self, batch_size, device):
        return self.init_state(batch_size, device)

    def sample_step(self, x, state):
        x_t = F.embedding(x.squeeze(1), self.embedding)
        h_f, c_f = self.forward_cell(x_t, state)
        backward_stub = torch.zeros_like(h_f)
        combined = torch.cat([h_f, backward_stub], dim=1)
        logits = (combined @ self.W_out + self.b_out).unsqueeze(1)
        return logits, (h_f, c_f)

blstm_model = BLSTM(vocab.vocab_size, HIDDEN_SIZE)
print(f"(ii) BLSTM Trainable Parameters: {count_parameters(blstm_model)}")

(ii) BLSTM Trainable Parameters: 75119


### BLSTM

## 3. RNN with Basic Attention Mechanism


### (iii) Hyperparameters
* **Hidden Size**: 64
* **Number of Layers**: 1
* **Learning Rate**: 0.005
* **Attention Type**: Bahdanau (Additive)

In [40]:
class AttentionRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Parameter(torch.randn(vocab_size, hidden_size) * 0.1)

        self.W_z = nn.Parameter(torch.randn(hidden_size, hidden_size) / np.sqrt(hidden_size))
        self.U_z = nn.Parameter(torch.randn(hidden_size, hidden_size) / np.sqrt(hidden_size))
        self.b_z = nn.Parameter(torch.zeros(hidden_size))

        self.W_r = nn.Parameter(torch.randn(hidden_size, hidden_size) / np.sqrt(hidden_size))
        self.U_r = nn.Parameter(torch.randn(hidden_size, hidden_size) / np.sqrt(hidden_size))
        self.b_r = nn.Parameter(torch.zeros(hidden_size))

        self.W_h = nn.Parameter(torch.randn(hidden_size, hidden_size) / np.sqrt(hidden_size))
        self.U_h = nn.Parameter(torch.randn(hidden_size, hidden_size) / np.sqrt(hidden_size))
        self.b_h = nn.Parameter(torch.zeros(hidden_size))

        self.W_query = nn.Parameter(torch.randn(hidden_size, hidden_size) / np.sqrt(hidden_size))
        self.W_key = nn.Parameter(torch.randn(hidden_size, hidden_size) / np.sqrt(hidden_size))
        self.b_attn = nn.Parameter(torch.zeros(hidden_size))
        self.v_attn = nn.Parameter(torch.randn(hidden_size, 1) / np.sqrt(hidden_size))

        self.W_out = nn.Parameter(torch.randn(hidden_size * 2, vocab_size) / np.sqrt(hidden_size * 2))
        self.b_out = nn.Parameter(torch.zeros(vocab_size))

    def init_hidden(self, batch_size, device):
        return torch.zeros(batch_size, self.hidden_size, device=device)

    def gru_step(self, x_t, h_prev):
        z = torch.sigmoid(x_t @ self.W_z + h_prev @ self.U_z + self.b_z)
        r = torch.sigmoid(x_t @ self.W_r + h_prev @ self.U_r + self.b_r)
        h_tilde = torch.tanh(x_t @ self.W_h + (r * h_prev) @ self.U_h + self.b_h)
        return (1 - z) * h_prev + z * h_tilde

    def attend(self, h_t, history):
        if not history:
            return torch.zeros_like(h_t)

        memory = torch.cat(history, dim=1)
        query = (h_t @ self.W_query).unsqueeze(1)
        keys = memory @ self.W_key
        scores = torch.tanh(query + keys + self.b_attn) @ self.v_attn
        weights = torch.softmax(scores.squeeze(-1), dim=1)
        context = torch.bmm(weights.unsqueeze(1), memory).squeeze(1)
        return context

    def forward(self, x):
        batch_size, seq_len = x.shape
        h = self.init_hidden(batch_size, x.device)
        history = []
        logits = []

        for t in range(seq_len):
            x_t = F.embedding(x[:, t], self.embedding)
            h = self.gru_step(x_t, h)
            context = self.attend(h, history)
            combined = torch.cat([h, context], dim=1)
            logits.append((combined @ self.W_out + self.b_out).unsqueeze(1))
            history.append(h.unsqueeze(1))

        return torch.cat(logits, dim=1)

    def init_generation_state(self, batch_size, device):
        return self.init_hidden(batch_size, device), []

    def sample_step(self, x, state):
        h_prev, history = state
        x_t = F.embedding(x.squeeze(1), self.embedding)
        h = self.gru_step(x_t, h_prev)
        context = self.attend(h, history)
        combined = torch.cat([h, context], dim=1)
        logits = (combined @ self.W_out + self.b_out).unsqueeze(1)
        new_history = history + [h.unsqueeze(1)]
        return logits, (h, new_history)

attn_model = AttentionRNN(vocab.vocab_size, HIDDEN_SIZE)
print(f"(ii) Attention RNN Trainable Parameters: {count_parameters(attn_model)}")

(ii) Attention RNN Trainable Parameters: 42159


### Attention RNN

## Training, Generation & Quantitative Evaluation Pipeline

In [41]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

vanilla_model = vanilla_model.to(device)
blstm_model = blstm_model.to(device)
attn_model = attn_model.to(device)

def train_model(model, names, vocab, epochs=10, model_type='vanilla', learning_rate=0.005):
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        shuffled_names = names.copy()
        random.shuffle(shuffled_names)

        for name in shuffled_names:
            encoded = vocab.encode(name)
            inputs = torch.tensor(encoded[:-1], dtype=torch.long, device=device).unsqueeze(0)
            targets = torch.tensor(encoded[1:], dtype=torch.long, device=device).unsqueeze(0)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs.reshape(-1, vocab.vocab_size), targets.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(shuffled_names)
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}")

def generate_names(model, vocab, num_names=20, model_type='vanilla', max_len=20, temperature=0.9):
    model.eval()
    generated = []

    with torch.no_grad():
        for _ in range(num_names):
            input_idx = torch.tensor([[vocab.char2idx['<SOS>']]], dtype=torch.long, device=device)
            state = model.init_generation_state(batch_size=1, device=device)
            generated_chars = []

            for _ in range(max_len):
                logits, state = model.sample_step(input_idx, state)
                probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
                next_idx = torch.multinomial(probs, num_samples=1).item()

                if next_idx == vocab.char2idx['<EOS>']:
                    break

                generated_chars.append(vocab.idx2char[next_idx])
                input_idx = torch.tensor([[next_idx]], dtype=torch.long, device=device)

            name = ''.join(generated_chars).strip()
            if name:
                generated.append(name)

    return generated


In [42]:
# Training the Models

models_dict = {
    'Vanilla RNN': (vanilla_model, 'vanilla'),
    'BLSTM': (blstm_model, 'blstm'),
    'Attention RNN': (attn_model, 'attention')
}

subset_names = training_names[:200]

print("Starting training process...")

for model_name, (model, m_type) in models_dict.items():
    print(f"\n{'='*40}\nTraining {model_name}\n{'='*40}")
    train_model(model, subset_names, vocab, epochs=10, model_type=m_type)

print("\nTraining complete for all models!")

Starting training process...

Training Vanilla RNN
Epoch 1/10, Loss: 2.6328
Epoch 2/10, Loss: 2.1903
Epoch 3/10, Loss: 2.0850
Epoch 4/10, Loss: 2.0042
Epoch 5/10, Loss: 1.9391
Epoch 6/10, Loss: 1.8686
Epoch 7/10, Loss: 1.7822
Epoch 8/10, Loss: 1.7363
Epoch 9/10, Loss: 1.6572
Epoch 10/10, Loss: 1.6008

Training BLSTM
Epoch 1/10, Loss: 1.7428
Epoch 2/10, Loss: 0.4138
Epoch 3/10, Loss: 0.1355
Epoch 4/10, Loss: 0.0450
Epoch 5/10, Loss: 0.0197
Epoch 6/10, Loss: 0.0082
Epoch 7/10, Loss: 0.0042
Epoch 8/10, Loss: 0.0030
Epoch 9/10, Loss: 0.0023
Epoch 10/10, Loss: 0.0019

Training Attention RNN
Epoch 1/10, Loss: 2.5623
Epoch 2/10, Loss: 2.1530
Epoch 3/10, Loss: 2.0409
Epoch 4/10, Loss: 1.9286
Epoch 5/10, Loss: 1.7940
Epoch 6/10, Loss: 1.6839
Epoch 7/10, Loss: 1.5467
Epoch 8/10, Loss: 1.4183
Epoch 9/10, Loss: 1.2981
Epoch 10/10, Loss: 1.2263

Training complete for all models!


In [50]:
# Comparing the Models

results = {}

print("Starting evaluation and comparison...\n")

for model_name, (model, m_type) in models_dict.items():
    print(f"Evaluating {model_name}...")
    
    gen_samples = generate_names(model, vocab, num_names=100, model_type=m_type)
    
    novelty, diversity = evaluate_metrics(gen_samples, training_names)
    
    results[model_name] = {
        'novelty': novelty,
        'diversity': diversity,
        'samples': gen_samples[:10]
    }
    
    print(f"Novelty Rate: {novelty:.2f}%")
    print(f"Diversity: {diversity:.2f}%")
    print(f"Representative Samples: {gen_samples[:10]}\n")
    print("-" * 50)
    
print("\nFINAL COMPARISON SUMMARY:")
for model_name, data in results.items():
    print(f"{model_name:<15} | Novelty: {data['novelty']:>5.2f}% | Diversity: {data['diversity']:>5.2f}%")

Starting evaluation and comparison...

Evaluating Vanilla RNN...
Novelty Rate: 92.00%
Diversity: 99.00%
Representative Samples: ['Vishal', 'Shaduti', 'Avichinta', 'Virendra', 'Ruchi', 'Nandilap', 'Varubha', 'Sushi', 'Arjita', 'Jeeshik']

--------------------------------------------------
Evaluating BLSTM...
Novelty Rate: 97.00%
Diversity: 97.00%
Representative Samples: ['Col', 'Del', 'Gm', 'La', 'Slol', 'Yol', 'Dmp', 'Iul', 'Dol', 'Bavy']

--------------------------------------------------
Evaluating Attention RNN...
Novelty Rate: 57.00%
Diversity: 93.00%
Representative Samples: ['Andira', 'Akhili', 'Indira', 'Rati', 'Hadita', 'Dolwshatt', 'Naveen', 'Rujima', 'Jomjhut', 'Paankhi']

--------------------------------------------------

FINAL COMPARISON SUMMARY:
Vanilla RNN     | Novelty: 92.00% | Diversity: 99.00%
BLSTM           | Novelty: 97.00% | Diversity: 97.00%
Attention RNN   | Novelty: 57.00% | Diversity: 93.00%
